# LeetCode 1196 — How Many Apples Can You Put into the Basket bằng Python

**Link đề:** https://leetcode.com/problems/how-many-apples-can-you-put-into-the-basket/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy, Sorting

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def maxNumberOfApples(self, weight: List[int]) -> int:
        ...
```

## Mục tiêu học
1. Nhận ra bài toán **"chọn được nhiều món nhất với tổng ≤ sức chứa"** → **tham lam (greedy): lấy món nhẹ nhất trước**.
2. Hiểu `sorted` / `.sort()`, vòng lặp cộng dồn, `break`, `enumerate`.
3. Chứng minh greedy đúng bằng **lập luận đổi chỗ (exchange argument)** và kiểm chứng bằng brute force thử mọi tập con.
4. Biết các cách khác: **counting sort** (vì cân nặng ≤ 1000) và **min-heap** (`heapq`).
5. Trace từng bước, test đối chiếu, bài tập TODO với đáp án ở cuối.

## 1. Đề bài và ví dụ

Bạn có một cái giỏ chịu được tối đa **5000** đơn vị cân nặng. Cho mảng `weight`, trong đó `weight[i]` là cân nặng của quả táo thứ `i`.
Trả về **số quả táo nhiều nhất** có thể bỏ vào giỏ (tổng cân nặng các quả đã chọn ≤ 5000).

| `weight` | Kết quả | Vì sao |
|---|:---:|---|
| `[100, 200, 150, 1000]` | `4` | tổng 1450 ≤ 5000, lấy hết |
| `[900, 950, 800, 1000, 700, 800]` | `5` | tổng cả 6 quả = 5150 > 5000; bỏ quả nặng nhất (1000) → 4150 |

**Constraints (theo hiểu biết):** `1 <= weight.length <= 10^3`; `1 <= weight[i] <= 10^3`.

Câu hỏi then chốt: muốn **nhiều quả nhất** chứ không phải **nặng nhất** → nên chọn quả nào trước?

In [1]:
CAPACITY = 5000
w = [900, 950, 800, 1000, 700, 800]
print("Tổng tất cả:", sum(w), "> 5000 ?", sum(w) > CAPACITY)
print("Bỏ quả 1000:", sum(w) - 1000, "<= 5000 ?", sum(w) - 1000 <= CAPACITY)

Tổng tất cả: 5150 > 5000 ? True
Bỏ quả 1000: 4150 <= 5000 ? True


## 2. Nền tảng Python cần biết

### 2.1. `sum`, `len`, `max`, `min`

Các hàm có sẵn chạy trên list: `sum(list)` cộng hết, `len(list)` đếm phần tử, `max`/`min` lấy lớn/nhỏ nhất. Mỗi hàm duyệt cả list → `O(n)`.

In [2]:
w = [100, 200, 150, 1000]
print("sum =", sum(w), "| len =", len(w), "| max =", max(w), "| min =", min(w))

sum = 1450 | len = 4 | max = 1000 | min = 100


### 2.2. `sorted(list)` và `list.sort()`

- `sorted(a)` trả **list mới** đã sắp xếp tăng dần, `a` giữ nguyên.
- `a.sort()` sắp xếp **ngay trên `a`** (in-place), trả `None`.
- Thêm `reverse=True` để giảm dần. Độ phức tạp `O(n log n)`.

In [3]:
a = [900, 950, 800, 1000, 700, 800]
b_sorted = sorted(a)
print("sorted(a) =", b_sorted, "| a vẫn là", a)
a.sort()
print("sau a.sort():", a)
print("a.sort() trả về:", [3, 1, 2].sort())
print("giảm dần:", sorted(a, reverse=True))

sorted(a) = [700, 800, 800, 900, 950, 1000] | a vẫn là [900, 950, 800, 1000, 700, 800]
sau a.sort(): [700, 800, 800, 900, 950, 1000]
a.sort() trả về: None
giảm dần: [1000, 950, 900, 800, 800, 700]


### 2.3. Cộng dồn và `break`

Mẫu **cộng dồn**: giữ biến `total`, mỗi bước `total += x`. `break` thoát vòng `for` ngay lập tức.

In [4]:
total = 0
for x in [3, 4, 5, 6]:
    if total + x > 10:
        print(f"thêm {x} sẽ thành {total + x} > 10 -> break")
        break
    total += x
    print("thêm", x, "-> total =", total)
print("total cuối:", total)

thêm 3 -> total = 3
thêm 4 -> total = 7
thêm 5 sẽ thành 12 > 10 -> break
total cuối: 7


### 2.4. `enumerate` — vừa có chỉ số vừa có giá trị

In [5]:
for i, x in enumerate([700, 800, 800]):
    print("i =", i, "x =", x)

i = 0 x = 700
i = 1 x = 800
i = 2 x = 800


### 2.5. `itertools.combinations` — để làm brute force

`combinations(a, k)` sinh mọi cách chọn `k` phần tử (không quan tâm thứ tự). Số tập con của `n` phần tử là `2^n` — bùng nổ rất nhanh.

In [6]:
from itertools import combinations
print(list(combinations([1, 2, 3], 2)))
print("Số tập con khi n = 20:", 2 ** 20, "| n = 1000:", "2^1000 ≈ 10^301 (không thể)")

[(1, 2), (1, 3), (2, 3)]
Số tập con khi n = 20: 1048576 | n = 1000: 2^1000 ≈ 10^301 (không thể)


## 3. Cách 1 — Brute force: thử mọi tập con

Cách "chắc chắn đúng" khi chưa nghĩ ra mẹo: xét **mọi tập con**, tập nào tổng ≤ 5000 thì ghi nhận số phần tử, lấy lớn nhất.

Mẹo nhỏ để đỡ tốn: thử `k` từ lớn đến nhỏ, gặp `k` nào có một tổ hợp hợp lệ thì trả luôn.

In [7]:
from itertools import combinations

def apples_bruteforce(weight: list[int], capacity: int = 5000) -> int:
    n = len(weight)
    for k in range(n, -1, -1):                 # n, n-1, ..., 0
        for combo in combinations(weight, k):
            if sum(combo) <= capacity:
                return k
    return 0

print(apples_bruteforce([100, 200, 150, 1000]))
print(apples_bruteforce([900, 950, 800, 1000, 700, 800]))

4
5


`range(n, -1, -1)` đếm ngược từ `n` xuống `0` (điểm dừng `-1` không được lấy).

**Nút thắt:** số tổ hợp tới `2^n`. Với `n = 1000` là không tưởng. Ta cần quy tắc chọn thông minh hơn.

## 4. Insight — muốn nhiều quả nhất thì lấy quả **nhẹ nhất** trước

Trực giác: mỗi quả "tốn" một phần sức chứa. Mục tiêu là **đếm số quả**, nên quả nào cũng "đáng giá" như nhau (1 quả). Vậy nên chọn quả **rẻ nhất** (nhẹ nhất) để còn chỗ cho nhiều quả khác.

Quy tắc greedy:
1. Sắp xếp `weight` tăng dần.
2. Lần lượt bỏ vào giỏ khi còn chỗ; quả đầu tiên không vừa → dừng (các quả sau còn nặng hơn, chắc chắn không vừa).

Ví dụ 2 sau khi sắp xếp: `[700, 800, 800, 900, 950, 1000]`

In [8]:
w = sorted([900, 950, 800, 1000, 700, 800])
prefix = 0
for i, x in enumerate(w):
    prefix += x
    print(f"lấy {i + 1} quả nhẹ nhất: tổng = {prefix} {'<= 5000 OK' if prefix <= 5000 else '> 5000 QUÁ TẢI'}")

lấy 1 quả nhẹ nhất: tổng = 700 <= 5000 OK
lấy 2 quả nhẹ nhất: tổng = 1500 <= 5000 OK
lấy 3 quả nhẹ nhất: tổng = 2300 <= 5000 OK
lấy 4 quả nhẹ nhất: tổng = 3200 <= 5000 OK
lấy 5 quả nhẹ nhất: tổng = 4150 <= 5000 OK
lấy 6 quả nhẹ nhất: tổng = 5150 > 5000 QUÁ TẢI


Đáp án = số **tổng tiền tố** (prefix sum) của mảng đã sắp xếp mà vẫn ≤ 5000 → 5.

## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — sắp xếp:** `weight = sorted(weight)` (không làm hỏng input gốc).

**Đoạn 2 — cộng dồn có kiểm tra:** giữ `total` (tổng đã bỏ vào) và `count` (số quả).

**Đoạn 3 — điều kiện dừng:** nếu `total + x > 5000` thì `break`. Kiểm tra **trước khi cộng**, để `total` luôn hợp lệ.

In [9]:
def apples_greedy(weight: list[int], capacity: int = 5000) -> int:
    total = 0
    count = 0
    for x in sorted(weight):          # đoạn 1
        if total + x > capacity:      # đoạn 3: kiểm tra trước
            break
        total += x                    # đoạn 2
        count += 1
    return count

for w in [[100, 200, 150, 1000], [900, 950, 800, 1000, 700, 800], [5000], [5001 - 1, 1]]:
    print(w, "->", apples_greedy(w))

[100, 200, 150, 1000] -> 4
[900, 950, 800, 1000, 700, 800] -> 5
[5000] -> 1
[5000, 1] -> 1


Phiên bản "trừ dần sức chứa" — cùng ý tưởng, đôi khi dễ đọc hơn:

In [10]:
def apples_greedy_remaining(weight: list[int], capacity: int = 5000) -> int:
    remaining = capacity
    for i, x in enumerate(sorted(weight)):
        remaining -= x
        if remaining < 0:
            return i                  # quả thứ i (0-based) làm quá tải -> lấy được i quả
    return len(weight)                # lấy hết

print(apples_greedy_remaining([900, 950, 800, 1000, 700, 800]))
print(apples_greedy_remaining([100, 200, 150, 1000]))

5
4


### Cách 3 — Counting sort (vì `1 <= weight[i] <= 1000`)

Giá trị nhỏ và bị chặn → đếm số quả của từng cân nặng `1..1000`, rồi duyệt cân nặng tăng dần. Không cần sort `O(n log n)`; chi phí `O(n + W)` với `W = 1000`.

In [11]:
def apples_counting(weight: list[int], capacity: int = 5000, max_w: int = 1000) -> int:
    freq = [0] * (max_w + 1)
    for x in weight:
        freq[x] += 1
    count = 0
    remaining = capacity
    for w in range(1, max_w + 1):
        if freq[w] == 0:
            continue
        take = min(freq[w], remaining // w)   # lấy tối đa có thể ở mức cân nặng w
        count += take
        remaining -= take * w
        if take < freq[w]:                    # không lấy hết -> các mức nặng hơn càng không vừa
            break
    return count

print(apples_counting([900, 950, 800, 1000, 700, 800]))
print(apples_counting([100, 200, 150, 1000]))

5
4


`continue` bỏ qua phần còn lại của vòng hiện tại và nhảy sang vòng kế tiếp.

### Cách 4 — Min-heap với `heapq`

`heapq.heapify(a)` biến list thành **min-heap** trong `O(n)`; `heapq.heappop(a)` lấy phần tử nhỏ nhất trong `O(log n)`.
Lợi thế: nếu chỉ lấy được `k` quả, chi phí là `O(n + k log n)` — không phải sort toàn bộ.

In [12]:
import heapq

h = [900, 950, 800, 1000, 700, 800]
heapq.heapify(h)
print("heap (dạng list):", h, "| nhỏ nhất h[0] =", h[0])
print("pop:", heapq.heappop(h), heapq.heappop(h), "| còn:", sorted(h))

def apples_heap(weight: list[int], capacity: int = 5000) -> int:
    h = list(weight)          # copy để không sửa input
    heapq.heapify(h)
    count, total = 0, 0
    while h and total + h[0] <= capacity:
        total += heapq.heappop(h)
        count += 1
    return count

print(apples_heap([900, 950, 800, 1000, 700, 800]), apples_heap([100, 200, 150, 1000]))

heap (dạng list): [700, 900, 800, 1000, 950, 800] | nhỏ nhất h[0] = 700
pop: 700 800 | còn: [800, 900, 950, 1000]
5 4


## 6. TRACE từng bước

Ví dụ `weight = [900, 950, 800, 1000, 700, 800]`, sau khi sắp xếp `[700, 800, 800, 900, 950, 1000]`:

| i | x | total trước | total + x | ≤ 5000? | Hành động | count |
|:---:|---:|---:|---:|:---:|---|:---:|
| 0 | 700 | 0 | 700 | ✓ | lấy | 1 |
| 1 | 800 | 700 | 1500 | ✓ | lấy | 2 |
| 2 | 800 | 1500 | 2300 | ✓ | lấy | 3 |
| 3 | 900 | 2300 | 3200 | ✓ | lấy | 4 |
| 4 | 950 | 3200 | 4150 | ✓ | lấy | 5 |
| 5 | 1000 | 4150 | 5150 | ✗ | **break** | 5 |

In [13]:
def trace_apples(weight: list[int], capacity: int = 5000) -> int:
    ws = sorted(weight)
    print("sau sắp xếp:", ws)
    total = count = 0
    for i, x in enumerate(ws):
        ok = total + x <= capacity
        print(f"i={i} x={x:4d} total trước={total:4d} total+x={total + x:4d} -> {'lấy' if ok else 'QUÁ TẢI, break'}")
        if not ok:
            break
        total += x
        count += 1
    print("Kết quả:", count, "quả, tổng", total)
    return count

assert trace_apples([900, 950, 800, 1000, 700, 800]) == 5

sau sắp xếp: [700, 800, 800, 900, 950, 1000]
i=0 x= 700 total trước=   0 total+x= 700 -> lấy
i=1 x= 800 total trước= 700 total+x=1500 -> lấy
i=2 x= 800 total trước=1500 total+x=2300 -> lấy
i=3 x= 900 total trước=2300 total+x=3200 -> lấy
i=4 x= 950 total trước=3200 total+x=4150 -> lấy
i=5 x=1000 total trước=4150 total+x=5150 -> QUÁ TẢI, break
Kết quả: 5 quả, tổng 4150


Trace thêm trường hợp **vừa khít** 5000 (dấu `<=` quan trọng) và trường hợp lấy hết:

In [14]:
assert trace_apples([1000, 3000, 1000, 1000, 1000, 1000]) == 5   # tổng đúng 5000 vẫn lấy
print()
assert trace_apples([100, 200, 150, 1000]) == 4

sau sắp xếp: [1000, 1000, 1000, 1000, 1000, 3000]
i=0 x=1000 total trước=   0 total+x=1000 -> lấy
i=1 x=1000 total trước=1000 total+x=2000 -> lấy
i=2 x=1000 total trước=2000 total+x=3000 -> lấy
i=3 x=1000 total trước=3000 total+x=4000 -> lấy
i=4 x=1000 total trước=4000 total+x=5000 -> lấy
i=5 x=3000 total trước=5000 total+x=8000 -> QUÁ TẢI, break
Kết quả: 5 quả, tổng 5000

sau sắp xếp: [100, 150, 200, 1000]
i=0 x= 100 total trước=   0 total+x= 100 -> lấy
i=1 x= 150 total trước= 100 total+x= 250 -> lấy
i=2 x= 200 total trước= 250 total+x= 450 -> lấy
i=3 x=1000 total trước= 450 total+x=1450 -> lấy
Kết quả: 4 quả, tổng 1450


## 7. Code nộp LeetCode

### 7.1. Bản chính — sort + greedy

In [15]:
from typing import List

class Solution:
    def maxNumberOfApples(self, weight: List[int]) -> int:
        total = 0
        count = 0
        for x in sorted(weight):
            if total + x > 5000:
                break
            total += x
            count += 1
        return count

sol = Solution()
print(sol.maxNumberOfApples([100, 200, 150, 1000]), sol.maxNumberOfApples([900, 950, 800, 1000, 700, 800]))

4 5


`from typing import List` cho phép viết gợi ý kiểu `List[int]` như skeleton LeetCode (trên LeetCode đã import sẵn). Từ Python 3.9 có thể viết `list[int]`.

### 7.2. Bản counting sort

In [16]:
class SolutionCounting:
    def maxNumberOfApples(self, weight: List[int]) -> int:
        freq = [0] * 1001
        for x in weight:
            freq[x] += 1
        count, remaining = 0, 5000
        for w in range(1, 1001):
            take = min(freq[w], remaining // w)
            count += take
            remaining -= take * w
            if take < freq[w]:
                break
        return count

assert SolutionCounting().maxNumberOfApples([900, 950, 800, 1000, 700, 800]) == 5
print("SolutionCounting OK")

SolutionCounting OK


### 7.3. Bản heap

In [17]:
import heapq

class SolutionHeap:
    def maxNumberOfApples(self, weight: List[int]) -> int:
        h = list(weight)
        heapq.heapify(h)
        count, total = 0, 0
        while h and total + h[0] <= 5000:
            total += heapq.heappop(h)
            count += 1
        return count

assert SolutionHeap().maxNumberOfApples([100, 200, 150, 1000]) == 4
print("SolutionHeap OK")

SolutionHeap OK


## 8. Vì sao greedy đúng? (exchange argument)

Gọi `s_1 ≤ s_2 ≤ ... ≤ s_n` là cân nặng sau khi sắp xếp. Greedy lấy `k` quả nhẹ nhất: `s_1 + ... + s_k ≤ 5000` nhưng `s_1 + ... + s_{k+1} > 5000`.

**Khẳng định:** không tập nào có `k + 1` quả mà tổng ≤ 5000.

**Lý do:** xét bất kỳ tập `T` gồm `k + 1` quả. Sắp các quả trong `T` tăng dần: `t_1 ≤ ... ≤ t_{k+1}`. Vì `s_1..s_{k+1}` là `k+1` giá trị **nhỏ nhất** của cả mảng, nên `t_j ≥ s_j` với mọi `j`. Cộng lại: `sum(T) ≥ s_1 + ... + s_{k+1} > 5000`. Vậy `T` quá tải. ∎

Nói cách khác (đổi chỗ): trong một lời giải tối ưu bất kỳ, nếu có quả nặng `y` được chọn mà quả nhẹ hơn `x < y` bị bỏ, đổi `y` lấy `x` — số quả giữ nguyên, tổng giảm, vẫn hợp lệ. Đổi mãi sẽ được đúng tập "k quả nhẹ nhất".

**Bất biến vòng lặp:** trước mỗi vòng `i`, `total = s_0 + ... + s_{i-1} ≤ 5000` và `count = i`.

Kiểm chứng thực nghiệm: so greedy với brute force trên hàng trăm mảng nhỏ ngẫu nhiên.

In [18]:
import random

rng = random.Random(1196)
for _ in range(300):
    n = rng.randint(1, 10)
    w = [rng.randint(1, 1000) for _ in range(n)]
    cap = rng.randint(1, 5000)
    assert apples_greedy(w, cap) == apples_bruteforce(w, cap), (w, cap)
print("Greedy == brute force trên 300 bộ ngẫu nhiên (sức chứa ngẫu nhiên)")

Greedy == brute force trên 300 bộ ngẫu nhiên (sức chứa ngẫu nhiên)


## 9. Độ phức tạp và so sánh

Gọi `n = len(weight)`, `W = 1000` là cân nặng tối đa, `k` = đáp án.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Brute force tập con | `O(2^n · n)` | `O(n)` | chỉ dùng làm oracle với `n` nhỏ |
| Sort + greedy | `O(n log n)` | `O(n)` (`sorted` tạo list mới; dùng `.sort()` để `O(1)` thêm nếu được sửa input) | cách chuẩn |
| Counting sort | `O(n + W)` | `O(W)` | tận dụng giá trị nhỏ |
| Min-heap | `O(n + k log n)` | `O(n)` | tốt khi `k` nhỏ hơn nhiều so với `n` |

Lưu ý: vì tổng ≤ 5000 và mỗi quả ≥ 1, `k ≤ 5000`; thực tế `n ≤ 1000` nên mọi cách (trừ brute force) đều rất nhanh.

## 10. Bộ test

Gồm mẫu, biên (1 quả, đúng 5000, mọi quả đều nặng), trùng lặp, và đối chiếu các cách trên input lớn ngẫu nhiên.

In [19]:
impls = {
    "greedy": apples_greedy,
    "remaining": apples_greedy_remaining,
    "counting": apples_counting,
    "heap": apples_heap,
    "Solution": Solution().maxNumberOfApples,
    "SolCounting": SolutionCounting().maxNumberOfApples,
    "SolHeap": SolutionHeap().maxNumberOfApples,
}
cases = [
    ([100, 200, 150, 1000], 4),
    ([900, 950, 800, 1000, 700, 800], 5),
    ([1], 1),                                   # 1 quả
    ([1000] * 5, 5),                            # vừa khít 5000
    ([1000] * 6, 5),                            # thừa đúng 1 quả
    ([1000] * 5 + [1], 5),                      # 1 + 4*1000 = 4001, thêm 1000 = 5001
    ([1] * 1000, 1000),                         # n tối đa, lấy hết
    ([999, 1, 1000, 2], 4),
    ([1000] * 1000, 5),
]
for w, expected in cases:
    for name, f in impls.items():
        assert f(w) == expected, (name, w[:8], f(w), expected)
    print(f"{str(w[:6]) + ('...' if len(w) > 6 else ''):40} -> {expected} PASS")

[100, 200, 150, 1000]                    -> 4 PASS
[900, 950, 800, 1000, 700, 800]          -> 5 PASS
[1]                                      -> 1 PASS
[1000, 1000, 1000, 1000, 1000]           -> 5 PASS
[1000, 1000, 1000, 1000, 1000, 1000]     -> 5 PASS
[1000, 1000, 1000, 1000, 1000, 1]        -> 5 PASS
[1, 1, 1, 1, 1, 1]...                    -> 1000 PASS
[999, 1, 1000, 2]                        -> 4 PASS
[1000, 1000, 1000, 1000, 1000, 1000]...  -> 5 PASS


In [20]:
rng = random.Random(51)
for _ in range(300):
    w = [rng.randint(1, 1000) for _ in range(rng.randint(1, 1000))]
    res = {name: f(w) for name, f in impls.items()}
    assert len(set(res.values())) == 1, res
w0 = [5, 3, 8]
Solution().maxNumberOfApples(w0)
assert w0 == [5, 3, 8], "không được sửa input"
print("300 mảng ngẫu nhiên: các cách khớp nhau; input không bị sửa")

300 mảng ngẫu nhiên: các cách khớp nhau; input không bị sửa


## 11. Lỗi người mới hay gặp

### Lỗi 1: Không sắp xếp, lấy theo thứ tự xuất hiện

Sai vì một quả nặng ở đầu chiếm chỗ của nhiều quả nhẹ phía sau.

In [21]:
def wrong_no_sort(weight):
    total = count = 0
    for x in weight:
        if total + x > 5000:
            break
        total += x
        count += 1
    return count

w = [1000, 1000, 1000, 1000, 999, 1, 1, 1]
print("không sort:", wrong_no_sort(w), "| đúng:", apples_greedy(w))

không sort: 6 | đúng: 7


### Lỗi 2: Sắp xếp giảm dần (lấy quả nặng trước)

Nhầm giữa "nhiều quả" với "nặng nhất". Lấy nặng trước làm **ít** quả hơn.

In [22]:
def wrong_desc(weight):
    total = count = 0
    for x in sorted(weight, reverse=True):
        if total + x > 5000:
            break
        total += x
        count += 1
    return count

w = [2000, 2000, 500, 500, 500, 500]
print("giảm dần:", wrong_desc(w), "| đúng:", apples_greedy(w))

giảm dần: 4 | đúng: 5


### Lỗi 3: Dùng `<` thay vì `<=` khi so với 5000

Giỏ chịu được **tối đa** 5000 → tổng **bằng** 5000 vẫn hợp lệ.

In [23]:
def wrong_strict(weight):
    total = count = 0
    for x in sorted(weight):
        if total + x >= 5000:      # SAI: loại luôn trường hợp đúng bằng 5000
            break
        total += x
        count += 1
    return count

print("strict:", wrong_strict([1000] * 5), "| đúng:", apples_greedy([1000] * 5))

strict: 4 | đúng: 5


### Lỗi 4: Cộng trước rồi mới kiểm tra, quên trả lại

```python
total += x
if total > 5000: break
count += 1
```

Cách này vẫn cho `count` đúng (vì `count` chưa tăng), nhưng `total` cuối cùng **sai** (đã cộng cả quả không vừa). Nếu đề hỏi thêm "tổng cân nặng trong giỏ" thì sẽ lỗi. Thói quen tốt: **kiểm tra trước, cập nhật sau**.

### Lỗi 5: `weight.sort()` rồi `for x in weight.sort():`

`.sort()` trả `None` → `TypeError: 'NoneType' object is not iterable`.

In [24]:
try:
    for x in [3, 1, 2].sort():
        pass
except TypeError as e:
    print("TypeError:", e)

TypeError: 'NoneType' object is not iterable


## 12. Mẹo nhận dạng pattern

- **"Tối đa số món"** + **"tổng ≤ giới hạn"** + **mỗi món có giá trị như nhau (đếm 1)** → sort tăng dần, lấy dần (greedy).
- Nếu mỗi món có **giá trị khác nhau** (ví dụ tối đa *tổng giá trị*) → đó là **knapsack**, greedy theo cân nặng **không** còn đúng, cần DP.
- Giá trị nhỏ, bị chặn (≤ 1000) → có thể thay sort bằng **counting sort**.
- Chỉ cần vài phần tử nhỏ nhất → nghĩ tới **heap**.

Bài cùng họ: 1833 Maximum Ice Cream Bars (y hệt, sức chứa là `coins`), 2389 Longest Subsequence With Limited Sum, 881 Boats to Save People (greedy + 2 con trỏ).

## 13. Biến thể

### 13.1. Sức chứa tuỳ ý — Maximum Ice Cream Bars (LeetCode 1833)

Chỉ đổi 5000 thành `coins`. Hàm `apples_greedy(weight, capacity)` đã hỗ trợ sẵn.

In [25]:
print(apples_greedy([1, 3, 2, 4, 1], 7))      # 4
print(apples_greedy([10, 6, 8, 7, 7, 8], 5))  # 0

4
0


### 13.2. Nhiều truy vấn sức chứa — prefix sum + binary search (LeetCode 2389)

Với nhiều giá trị `cap`, sort một lần, tính **tổng tiền tố**, mỗi truy vấn dùng `bisect_right(prefix, cap)` để đếm có bao nhiêu tổng tiền tố ≤ cap.
`itertools.accumulate` tạo dãy tổng tiền tố; `bisect_right(a, x)` trả số phần tử `≤ x` trong list đã sắp xếp `a` trong `O(log n)`.

In [26]:
from itertools import accumulate
from bisect import bisect_right

def answer_queries(nums: list[int], queries: list[int]) -> list[int]:
    prefix = list(accumulate(sorted(nums)))
    return [bisect_right(prefix, q) for q in queries]

print(list(accumulate([1, 2, 4, 5])))
print(answer_queries([4, 5, 2, 1], [3, 10, 21]))   # [2, 3, 4]

[1, 3, 7, 12]
[2, 3, 4]


### 13.3. Hai giỏ, mỗi giỏ 5000

Không còn đơn giản là lấy prefix — phải chia quả vào 2 giỏ (bài toán bin packing, khó hơn nhiều). Đây là ví dụ cho thấy greedy chỉ đúng với **đúng cấu trúc bài**; thay đổi nhỏ có thể phá vỡ nó.

## 14. Bài tập (TODO — không gọi khi chạy notebook)

**Bài 1 (tính tay).** `weight = [400, 4000, 300, 200, 100]`. Sắp xếp, lập bảng tổng tiền tố, cho biết đáp án.

**Bài 2 (dự đoán output).** `sorted([3, 1, 2])`, `[3, 1, 2].sort()`, `list(range(3, -1, -1))`.

**Bài 3 (điền TODO).** Hoàn thiện greedy dùng biến `remaining`.

In [27]:
def exercise_apples(weight: list[int]) -> int:
    remaining = 5000
    count = 0
    # TODO 1: duyệt x trong weight đã sắp xếp tăng dần
    # TODO 2: nếu x > remaining thì dừng
    # TODO 3: trừ remaining, tăng count
    pass

# assert exercise_apples([900, 950, 800, 1000, 700, 800]) == 5
# assert exercise_apples([1000] * 5) == 5

**Bài 4 (viết lại).** Viết lời giải không dùng `sorted`/`.sort()` mà dùng **counting sort** với `freq` 1001 ô.

In [28]:
def exercise_apples_counting(weight: list[int]) -> int:
    # TODO
    pass

# assert exercise_apples_counting([100, 200, 150, 1000]) == 4

**Bài 5 (biến thể).** Trả về **tổng cân nặng** của giỏ khi đã chọn được nhiều quả nhất (nếu nhiều cách cùng số quả, lấy tổng **nhỏ nhất** — greedy đã cho đúng tổng nhỏ nhất).

In [29]:
def exercise_apples_weight(weight: list[int]) -> int:
    # TODO
    pass

# assert exercise_apples_weight([900, 950, 800, 1000, 700, 800]) == 4150

**Bài 6 (mini challenge).** Boats to Save People (LeetCode 881): mỗi thuyền chở **tối đa 2 người**, tổng ≤ `limit`. Trả số thuyền **ít nhất**. Gợi ý: sort + hai con trỏ (người nhẹ nhất ghép với người nặng nhất nếu được).

In [30]:
def exercise_num_boats(people: list[int], limit: int) -> int:
    # TODO
    pass

# assert exercise_num_boats([1, 2], 3) == 1
# assert exercise_num_boats([3, 2, 2, 1], 3) == 3
# assert exercise_num_boats([3, 5, 3, 4], 5) == 4

## 15. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Nhẹ trước | sorted rồi cộng dồn | tổng tiền tố là 100, 300, 600, ... — dừng ở đâu? |
| 2 | Phân biệt trả list mới / trả None | `range(start, stop, step)` | stop không được lấy |
| 3 | Kiểm tra trước, trừ sau | `for x in sorted(weight)` | `if x > remaining: break` |
| 4 | Đếm số quả theo từng cân nặng | `for w in range(1, 1001)` | `take = min(freq[w], remaining // w)` |
| 5 | Giữ thêm biến `total` | trả `total` thay vì `count` | `total += x` trong nhánh lấy |
| 6 | Người nặng nhất luôn phải đi; đi cùng ai? | `i, j = 0, n - 1` | `if p[i] + p[j] <= limit: i += 1`; luôn `j -= 1; boats += 1` |

## 16. Đáp án tham khảo

**Bài 1:** sau sort `[100, 200, 300, 400, 4000]`, tiền tố `100, 300, 600, 1000, 5000` → cả 5 ≤ 5000 → **5** (vừa khít).

**Bài 2:** `[1, 2, 3]`, `None`, `[3, 2, 1, 0]`.

In [31]:
def answer_apples(weight: list[int]) -> int:
    remaining = 5000
    count = 0
    for x in sorted(weight):
        if x > remaining:
            break
        remaining -= x
        count += 1
    return count

def answer_apples_counting(weight: list[int]) -> int:
    freq = [0] * 1001
    for x in weight:
        freq[x] += 1
    count, remaining = 0, 5000
    for w in range(1, 1001):
        take = min(freq[w], remaining // w)
        count += take
        remaining -= take * w
        if take < freq[w]:
            break
    return count

def answer_apples_weight(weight: list[int]) -> int:
    total = 0
    for x in sorted(weight):
        if total + x > 5000:
            break
        total += x
    return total

def answer_num_boats(people: list[int], limit: int) -> int:
    p = sorted(people)
    i, j = 0, len(p) - 1
    boats = 0
    while i <= j:
        if p[i] + p[j] <= limit:
            i += 1
        j -= 1
        boats += 1
    return boats

assert apples_greedy([400, 4000, 300, 200, 100]) == 5
assert (sorted([3, 1, 2]), [3, 1, 2].sort(), list(range(3, -1, -1))) == ([1, 2, 3], None, [3, 2, 1, 0])
assert answer_apples([900, 950, 800, 1000, 700, 800]) == 5
assert answer_apples([1000] * 5) == 5
assert answer_apples_counting([100, 200, 150, 1000]) == 4
assert answer_apples_weight([900, 950, 800, 1000, 700, 800]) == 4150
assert answer_num_boats([1, 2], 3) == 1
assert answer_num_boats([3, 2, 2, 1], 3) == 3
assert answer_num_boats([3, 5, 3, 4], 5) == 4
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
def max_items(costs, budget):
    count = 0
    for c in sorted(costs):          # rẻ nhất trước
        if c > budget:               # kiểm tra trước
            break
        budget -= c                  # cập nhật sau
        count += 1
    return count
```

**Câu thần chú:** *"Đếm số món, mỗi món như nhau → lấy món rẻ nhất trước."*

### Checklist trước khi submit
- [ ] Sắp xếp **tăng dần** (không phải giảm dần, không quên sort)?
- [ ] So sánh dùng `<=` 5000 (tổng bằng 5000 vẫn hợp lệ)?
- [ ] Trường hợp lấy hết mảng trả đúng `len(weight)`?
- [ ] Không viết `for x in weight.sort()`?
- [ ] Tên method `maxNumberOfApples(self, weight: List[int]) -> int`, có `from typing import List` khi chạy ngoài LeetCode?